# Bellwether — candidate pool + validation (Prompt 8, Phase 3 payoff)

Build a filtered, ranked candidate pool; score how copyable each wallet is
(input filter to Experiment A); validate the ranking out-of-sample with a
walk-forward back-test + shuffled-label control.

**Note:** a real >=500-wallet pool with >=50 resolved trades each needs a broad
load (leaderboard/seed + per-wallet backfills) and wallets whose markets have
resolved. The math is validated on fixtures in `tests/test_candidates.py`.

In [ ]:
from bellwether_analytics.candidates import PoolConfig, build_pool, persist_pool, trackability_score, walk_forward
from bellwether_analytics.core import load_events_df, load_trades_df

trades = load_trades_df(platform="polymarket")
events = load_events_df(platform="polymarket")
print(f"{len(trades)} trades, {trades['wallet'].nunique()} wallets")

In [ ]:
# Candidate pool. Use production thresholds (>=50 resolved, >=55% win) once enough
# resolved data is loaded; loosen for a prototype dataset.
pool = build_pool(trades, events, PoolConfig(min_resolved_trades=0, min_win_rate=0.0))
pool.head(25)

In [ ]:
# Trackability pre-score (input filter to Experiment A): slow/deep > fast/thin.
trackability_score(trades).head(25)

In [ ]:
# Walk-forward out-of-sample validation + shuffled-label control.
# edge > 0 and edge >> edge_shuffled means the ranking has real predictive power.
walk_forward(trades, split_ts="2026-03-01")

In [ ]:
# Persist the pool to candidate_score (the dashboard leaderboard reads this).
persist_pool(pool, platform="polymarket")